# Unification of published files and contracts



In [1]:
import pandas as pd
import numpy as np
from pyprojroot import here
import utils
processed_data = here('data/processed_data')

# Load Contracts

In [2]:
contracts = pd.read_feather(processed_data / '1_2_merged_contracts_preprocessed.feather')

In [3]:
#### NON VALID ROWS #############################
print(contracts.shape)
contracts = contracts[contracts['supplier_name_clean'] != '#name?'].reset_index(drop=True)
contracts = contracts[contracts['supplier_name_clean'] != ''].reset_index(drop=True)
contracts = contracts[contracts['supplier_name_clean'].isnull() == False].reset_index(drop=True)
print(contracts.shape)

contracts['contract_price_mx'] = np.where(contracts['contract_price_mx'] <= 0, np.nan, contracts['contract_price_mx'])
print(contracts['contract_price_mx'].isnull().sum())
contracts = contracts.dropna(subset=['contract_price_mx']).reset_index(drop=True)
print(contracts['contract_price_mx'].isnull().sum())



(2315321, 32)
(2315215, 32)
1372
0


In [4]:
contracts.columns

Index(['government_level', 'purchasing_unit_name', 'file_code',
       'file_template', 'procedure_number', 'award_date', 'advertisement_date',
       'submission_deadline_date', 'country_contract_type', 'contracting_type',
       'procedure_type', 'procedure_venue', 'contract_code',
       'contract_initial_date', 'contract_end_date', 'contract_price_mx',
       'contract_signature_date', 'supplier_type_alternative',
       'providers_registry_code', 'supplier_name', 'supplier_type',
       'advertisement_url', 'df_year', 'legal_fundament', 'RFC',
       'purchasing_unit_id', 'supply_type', 'legal_framework', 'supplier_size',
       'supplier_name_clean', 'legal_fundament_standard',
       'legal_fundament_simplified'],
      dtype='object')

In [5]:
contracts['contract_year'] = contracts['contract_initial_date'].dt.year.astype(int)

In [6]:
contracts['contract_year'].value_counts()

2017    234067
2016    231246
2015    220647
2021    209886
2019    196676
2014    195591
2018    194078
2022    193771
2013    177946
2012    167311
2020    163656
2011    124033
2010      4935
Name: contract_year, dtype: int64

In [7]:
#drop 2010
contracts = contracts[contracts['contract_year'] != 2010].reset_index(drop=True)

In [8]:
nrows_contracts_original = contracts.shape[0]

In [9]:
nrows_contracts_original

2308908

In [10]:
contracts['contract_code'] = contracts['contract_code'].astype(str)

In [11]:
#add to all columns the ending _c
contracts.columns = [str(col) + '_c' for col in contracts.columns]

# Load and merge published files

In [12]:
variables2keep = ['file_code', 'advertisement_date', 'state', 'advertisement_link', 'submission_deadline_date'] #i realized that the other values are unreliable

In [13]:
published_files = pd.read_csv(processed_data / 'minimal_procedures_merged.csv', encoding='utf-8', sep=',', usecols=variables2keep)

/mnt/sdb1/marti/miniconda3/envs/det_corr_mex_env/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3257: DtypeWarning: Columns (0,12) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [14]:
published_files.isnull().sum()  / published_files.shape[0]

file_code                   0.000000e+00
state                       5.382618e-07
advertisement_date          1.922833e-02
advertisement_link          0.000000e+00
submission_deadline_date    6.858338e-01
dtype: float64

In [16]:
len(published_files)

1857832

In [17]:
published_files = published_files.drop_duplicates()

In [18]:
len(published_files)

1847730

In [20]:
published_files['advertisement_date'] = pd.to_datetime(published_files['advertisement_date'], errors='raise')
published_files['advertisement_link'] = published_files['advertisement_link'].astype(str)
published_files['file_code'] = published_files['file_code'].astype(str)

In [21]:
contracts['advertisement_url_c'] = contracts['advertisement_url_c'].astype(str)
contracts['file_code_c'] = contracts['file_code_c'].astype(str)

In [22]:
published_files['submission_deadline_date'] = pd.to_datetime(published_files['submission_deadline_date'], errors='raise')

In [23]:
#add to all columns the ending _pf
published_files.columns = [str(col) + '_pf' for col in published_files.columns]

In [24]:
print(len(published_files))
print(len(published_files.drop_duplicates(subset=['file_code_pf', 'advertisement_link_pf'])))


1847730
1847729


In [25]:
published_files = published_files.drop_duplicates(subset=['file_code_pf', 'advertisement_link_pf']).reset_index(drop=True)

In [26]:
published_files.shape

(1847729, 5)

In [27]:
len(contracts)

2308908

In [28]:
contracts_and_procedure = pd.merge(contracts, published_files, how='left', left_on=['file_code_c', 'advertisement_url_c'], right_on=['file_code_pf', 'advertisement_link_pf'])


In [29]:
#total contracts
len(contracts_and_procedure)

2308908

### for the paper how many matches are there

In [30]:
#MEASURE HOW MANY MATCHES ARE THERE
contracts_base = contracts.copy()
print(len(contracts_base))
contracts_base = contracts['contract_year_c'].value_counts().reset_index(name='count')
published_files_base = published_files[['file_code_pf', 'advertisement_link_pf']].drop_duplicates()
published_files_base['match'] = 1
print(len(published_files_base))
merging_prop = contracts.merge(published_files_base, how='left', left_on = ['file_code_c', 'advertisement_url_c'] , right_on=['file_code_pf', 'advertisement_link_pf'])
merging_prop = merging_prop.dropna(subset=['match'])
print(len(merging_prop))
merging_prop = merging_prop['contract_year_c'].value_counts().reset_index(name = 'count')


2308908
1847729
1341345


In [31]:
merging_prop = contracts_base.merge(merging_prop, how='left', on='index')
merging_prop['prop'] = merging_prop['count_y'] / merging_prop['count_x']
merging_prop = merging_prop.rename(columns={
    'index': 'Contract Year',
    'count_x': 'N. c-Compranet',
    'count_y': 'N. p-Compranet',
    'prop': 'Prop. of Matches'})

#sort by contract year
merging_prop = merging_prop.sort_values(by='Contract Year')
#to latex
latex_str = merging_prop.to_latex(index=False)
print(latex_str)


\begin{tabular}{rrrr}
\toprule
 Contract Year &  N. c-Compranet &  N. p-Compranet &  Prop. of Matches \\
\midrule
          2011 &          124033 &              23 &          0.000185 \\
          2012 &          167311 &             241 &          0.001440 \\
          2013 &          177946 &          107851 &          0.606088 \\
          2014 &          195591 &          119697 &          0.611976 \\
          2015 &          220647 &          158860 &          0.719974 \\
          2016 &          231246 &          188391 &          0.814678 \\
          2017 &          234067 &          211039 &          0.901618 \\
          2018 &          194078 &            6736 &          0.034708 \\
          2019 &          196676 &          177663 &          0.903328 \\
          2020 &          163656 &            6194 &          0.037848 \\
          2021 &          209886 &          185776 &          0.885128 \\
          2022 &          193771 &          178874 &          0.923121 \

In [32]:
merging_prop['N. p-Compranet'].sum()

1341345

# Loard APF dataset and merge

In [33]:
#import procedures_pdn
extended_procedures = pd.read_csv(processed_data / 'procedures_apf_clean.csv', encoding='latin-1')
extended_procedures.columns = [str(col) + '_p' for col in extended_procedures.columns]

In [34]:
len(extended_procedures)

957581

In [35]:
extended_procedures.columns

Index(['tender_id_p', 'tender_numberOfTenderers_p',
       'tender_procurementMethod_p', 'tender_tenderPeriod_startDate_p',
       'tender_tenderPeriod_endDate_p', 'tender_awardPeriod_endDate_p',
       'awards_contractPeriod_startDate_p', 'awards_id_p',
       'awards_per_tender_p', 'buyer_parties_contactPoint_name_clean_p',
       'buyer_parties_roles_p'],
      dtype='object')

In [36]:
extended_procedures['tender_tenderPeriod_startDate_p'] = pd.to_datetime(extended_procedures['tender_tenderPeriod_startDate_p'], errors='raise')
extended_procedures['tender_tenderPeriod_endDate_p'] = pd.to_datetime(extended_procedures['tender_tenderPeriod_endDate_p'], errors='raise')
extended_procedures['tender_awardPeriod_endDate_p'] = pd.to_datetime(extended_procedures['tender_awardPeriod_endDate_p'], errors='raise')
extended_procedures['awards_contractPeriod_startDate_p'] = pd.to_datetime(extended_procedures['awards_contractPeriod_startDate_p'], errors='raise')

In [37]:
extended_procedures['awards_id_p'] = extended_procedures['awards_id_p'].astype(str)

In [38]:
#missing values
extended_procedures.isnull().sum() / len(extended_procedures)

tender_id_p                                0.000000
tender_numberOfTenderers_p                 0.657975
tender_procurementMethod_p                 0.000056
tender_tenderPeriod_startDate_p            0.000000
tender_tenderPeriod_endDate_p              0.834798
tender_awardPeriod_endDate_p               0.000095
awards_contractPeriod_startDate_p          0.000000
awards_id_p                                0.000000
awards_per_tender_p                        0.000000
buyer_parties_contactPoint_name_clean_p    0.000065
buyer_parties_roles_p                      0.000065
dtype: float64

In [39]:
len(extended_procedures.drop_duplicates(subset=['awards_id_p', 'tender_id_p', 'awards_contractPeriod_startDate_p']))

830841

In [40]:
len(extended_procedures.drop_duplicates(subset=['awards_id_p', 'awards_contractPeriod_startDate_p']))

830841

In [46]:
#MEASURE HOW MANY MATCHES ARE POSSIBLE BETWEEN CONTRACTS AND PROCEDURES GIVEN CONTRACT CODE AND INITIAL DATE
contracts_base = contracts.copy()
print(len(contracts_base))
contracts_base = contracts['contract_year_c'].value_counts().reset_index(name = 'count')
extended_procedures_basic = extended_procedures[['awards_id_p', 'awards_contractPeriod_startDate_p']].drop_duplicates()
extended_procedures_basic['match'] = 1
print(len(extended_procedures_basic))
merging_prop = contracts.merge(extended_procedures_basic, how='left', left_on = ['contract_code_c', 'contract_initial_date_c'] , right_on=['awards_id_p', 'awards_contractPeriod_startDate_p'])
merging_prop = merging_prop.dropna(subset=['match'])
print(len(merging_prop))
merging_prop = merging_prop['contract_year_c'].value_counts().reset_index(name = 'count')



2308908
830841
774254


In [47]:
merging_prop = contracts_base.merge(merging_prop, how='left', on='index')
merging_prop['prop'] = merging_prop['count_y'] / merging_prop['count_x']
merging_prop = merging_prop.rename(columns={
    'index': 'Contract Year',
    'count_x': 'N. c-Compranet',
    'count_y': 'N. p-Compranet',
    'prop': 'Prop. of Matches'})

#sort by contract year
merging_prop = merging_prop.sort_values(by='Contract Year')
merging_prop['N. p-Compranet'] = np.where(merging_prop['N. p-Compranet'].isnull(), 0, merging_prop['N. p-Compranet'])
merging_prop['N. p-Compranet'] = merging_prop['N. p-Compranet'].astype(int)
merging_prop['Prop. of Matches'] = np.where(merging_prop['Prop. of Matches'].isnull(), 0, merging_prop['Prop. of Matches'])
#to latex
latex_str = merging_prop.to_latex(index=False)
print(latex_str)


\begin{tabular}{rrrr}
\toprule
 Contract Year &  N. c-Compranet &  N. p-Compranet &  Prop. of Matches \\
\midrule
          2011 &          124033 &               0 &          0.000000 \\
          2012 &          167311 &               0 &          0.000000 \\
          2013 &          177946 &               1 &          0.000006 \\
          2014 &          195591 &               0 &          0.000000 \\
          2015 &          220647 &               1 &          0.000005 \\
          2016 &          231246 &              21 &          0.000091 \\
          2017 &          234067 &           23733 &          0.101394 \\
          2018 &          194078 &          144078 &          0.742372 \\
          2019 &          196676 &          164568 &          0.836747 \\
          2020 &          163656 &          127418 &          0.778572 \\
          2021 &          209886 &          151775 &          0.723131 \\
          2022 &          193771 &          162659 &          0.839439 \

In [48]:
merging_prop['N. p-Compranet'].sum()

774254

In [41]:
contracts_and_procedure['file_code_c'] = contracts_and_procedure['file_code_c'].astype(str)
extended_procedures['tender_id_p'] = extended_procedures['tender_id_p'].astype(str)

contracts_and_procedure['contract_code_c'] = contracts_and_procedure['contract_code_c'].astype(str)
extended_procedures['awards_id_p'] = extended_procedures['awards_id_p'].astype(str)

extended_procedures['awards_contractPeriod_startDate_p'] = pd.to_datetime(extended_procedures['awards_contractPeriod_startDate_p'], errors='raise')

In [42]:
#merge with contract code
#variables to take into account
key_column_l = ['file_code_c', 'contract_code_c', 'contract_initial_date_c']
key_column_r = ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
objective_column = ['tender_awardPeriod_endDate_p']
#df to merge
tmerging_df = utils.non_duplicate_df(source_df = extended_procedures, key_column = key_column_r, interest_column = objective_column, filter_duplicate_all = False, lower_to_higher=True)
#merging
contracts_and_procedure = pd.merge(contracts_and_procedure, tmerging_df, how='left', left_on=key_column_l, right_on=key_column_r)
contracts_and_procedure = contracts_and_procedure.drop(columns=key_column_r)
print(contracts_and_procedure.shape[0] == nrows_contracts_original)

len df:  834982
there are 834982 rows without nan and without duplicates.
We will delete  4213 rows that have different  ['tender_awardPeriod_endDate_p']  even with same  ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
We are keeping lower_to_higher =  True
True


In [43]:
#merge with contract code
#variables to take into account
key_column_l = ['file_code_c', 'contract_code_c', 'contract_initial_date_c']
key_column_r = ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
objective_column = ['tender_tenderPeriod_startDate_p']
#df to merge
tmerging_df = utils.non_duplicate_df(source_df = extended_procedures, key_column = key_column_r, interest_column = objective_column, filter_duplicate_all = False, lower_to_higher=True)
#merging
contracts_and_procedure = pd.merge(contracts_and_procedure, tmerging_df, how='left', left_on=key_column_l, right_on=key_column_r)
contracts_and_procedure = contracts_and_procedure.drop(columns=key_column_r)
print(contracts_and_procedure.shape[0] == nrows_contracts_original)

len df:  830843
there are 830843 rows without nan and without duplicates.
We will delete  2 rows that have different  ['tender_tenderPeriod_startDate_p']  even with same  ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
We are keeping lower_to_higher =  True
True


In [44]:
#tender_tenderPeriod_endDate_p
#variables to take into account
key_column_l = ['file_code_c', 'contract_code_c', 'contract_initial_date_c']
key_column_r = ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
objective_column = ['tender_tenderPeriod_endDate_p']
#df to merge
tmerging_df = utils.non_duplicate_df(source_df = extended_procedures, key_column = key_column_r, interest_column = objective_column, filter_duplicate_all = False, lower_to_higher=True)
#merging
contracts_and_procedure = pd.merge(contracts_and_procedure, tmerging_df, how='left', left_on=key_column_l, right_on=key_column_r)
contracts_and_procedure = contracts_and_procedure.drop(columns=key_column_r)
print(contracts_and_procedure.shape[0] == nrows_contracts_original)

len df:  144109
there are 144109 rows without nan and without duplicates.
We will delete  4 rows that have different  ['tender_tenderPeriod_endDate_p']  even with same  ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
We are keeping lower_to_higher =  True
True


In [45]:
#tender_numberOfTenderers_p
#variables to take into account
key_column_l = ['file_code_c', 'contract_code_c', 'contract_initial_date_c']
key_column_r = ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
objective_column = ['tender_numberOfTenderers_p']
#df to merge
tmerging_df = utils.non_duplicate_df(source_df = extended_procedures, key_column = key_column_r, interest_column = objective_column, filter_duplicate_all = False, lower_to_higher=False)
#merging
contracts_and_procedure = pd.merge(contracts_and_procedure, tmerging_df, how='left', left_on=key_column_l, right_on=key_column_r)
contracts_and_procedure = contracts_and_procedure.drop(columns=key_column_r)
print(contracts_and_procedure.shape[0] == nrows_contracts_original)

len df:  288362
there are 288362 rows without nan and without duplicates.
We will delete  558 rows that have different  ['tender_numberOfTenderers_p']  even with same  ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
We are keeping lower_to_higher =  False
True


In [46]:
#awards_per_tender
#variables to take into account
key_column_l = ['file_code_c', 'contract_code_c', 'contract_initial_date_c']
key_column_r = ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
objective_column = ['awards_per_tender_p']
#df to merge
tmerging_df = utils.non_duplicate_df(source_df = extended_procedures, key_column = key_column_r, interest_column = objective_column, filter_duplicate_all = False, lower_to_higher=False)
#merging
contracts_and_procedure = pd.merge(contracts_and_procedure, tmerging_df, how='left', left_on=key_column_l, right_on=key_column_r)
contracts_and_procedure = contracts_and_procedure.drop(columns=key_column_r)
print(contracts_and_procedure.shape[0] == nrows_contracts_original)

len df:  855955
there are 855955 rows without nan and without duplicates.
We will delete  25114 rows that have different  ['awards_per_tender_p']  even with same  ['tender_id_p', 'awards_id_p', 'awards_contractPeriod_startDate_p']
We are keeping lower_to_higher =  False
True


# Create unique variables with the information of all datasets

The priority in importance

Contracts >> published files >> apf dataset

The variables are :

advertisement_date
submission_deadline_date
award_date
contract_signature_date
number_of_tenders

### Advertisement date

In [47]:
contracts_and_procedure['advertisement_date'] = contracts_and_procedure['advertisement_date_c'].copy()
contracts_and_procedure['advertisement_date'] = np.where(contracts_and_procedure['advertisement_date'].isnull(), contracts_and_procedure['advertisement_date_pf'], contracts_and_procedure['advertisement_date'])
contracts_and_procedure['advertisement_date'] = np.where(contracts_and_procedure['advertisement_date'].isnull(), contracts_and_procedure['tender_tenderPeriod_startDate_p'], contracts_and_procedure['advertisement_date'])

In [48]:
contracts_and_procedure[[ 'advertisement_date', 'advertisement_date_c', 'advertisement_date_pf', 'tender_tenderPeriod_startDate_p']].isnull().sum() / len(contracts_and_procedure)

advertisement_date                 0.173678
advertisement_date_c               0.326794
advertisement_date_pf              0.419087
tender_tenderPeriod_startDate_p    0.664667
dtype: float64

In [49]:
#drop the columns we don't need
contracts_and_procedure = contracts_and_procedure.drop(columns=['advertisement_date_c', 'advertisement_date_pf', 'tender_tenderPeriod_startDate_p'])

### Submission deadline date

In [50]:
contracts_and_procedure['submission_deadline_date'] = contracts_and_procedure['submission_deadline_date_c'].copy()
contracts_and_procedure['submission_deadline_date'] = np.where(contracts_and_procedure['submission_deadline_date'].isnull(), contracts_and_procedure['submission_deadline_date_pf'], contracts_and_procedure['submission_deadline_date'])
contracts_and_procedure['submission_deadline_date'] = np.where(contracts_and_procedure['submission_deadline_date'].isnull(), contracts_and_procedure['tender_tenderPeriod_endDate_p'], contracts_and_procedure['submission_deadline_date'])


In [51]:
contracts_and_procedure[['submission_deadline_date', 'submission_deadline_date_c', 'submission_deadline_date_pf', 'tender_tenderPeriod_endDate_p']].isnull().sum() / len(contracts_and_procedure)

submission_deadline_date         0.282378
submission_deadline_date_c       0.285118
submission_deadline_date_pf      0.831666
tender_tenderPeriod_endDate_p    0.940486
dtype: float64

In [52]:
#drop the columns we don't need
contracts_and_procedure = contracts_and_procedure.drop(columns=['submission_deadline_date_c', 'tender_tenderPeriod_endDate_p', 'submission_deadline_date_pf'])

In [53]:
contracts_and_procedure['submission_deadline_date']

0         2011-12-15
1         2011-12-16
2         2011-12-16
3         2011-09-19
4         2011-09-13
             ...    
2308903          NaT
2308904          NaT
2308905          NaT
2308906          NaT
2308907          NaT
Name: submission_deadline_date, Length: 2308908, dtype: datetime64[ns]

### Award date

In [54]:
contracts_and_procedure['award_date_c'] = pd.to_datetime(contracts_and_procedure['award_date_c'], errors='raise')

In [55]:
contracts_and_procedure['award_date'] = contracts_and_procedure['award_date_c'].copy()
contracts_and_procedure['award_date'] = np.where(contracts_and_procedure['award_date'].isnull(), contracts_and_procedure['tender_awardPeriod_endDate_p'], contracts_and_procedure['award_date'])

In [56]:
contracts_and_procedure[['award_date', 'award_date_c', 'tender_awardPeriod_endDate_p']].isnull().sum() / len(contracts_and_procedure)

award_date                      0.595893
award_date_c                    0.871682
tender_awardPeriod_endDate_p    0.664696
dtype: float64

In [57]:
#drop the columns we don't need
contracts_and_procedure = contracts_and_procedure.drop(columns=['award_date_c', 'tender_awardPeriod_endDate_p'])

### Number of tenders

In [58]:
#change the name of the column from tender_numberOfTenderers_p to number_of_tenderers
contracts_and_procedure = contracts_and_procedure.rename(columns={'tender_numberOfTenderers_p': 'number_of_tenderers'})



In [59]:
contracts_and_procedure['number_of_tenderers'].isnull().sum() / len(contracts_and_procedure)

0.8792918557170749

### Awards per tender

In [60]:
contracts_and_procedure = contracts_and_procedure.rename(columns={'awards_per_tender_p': 'awards_per_tender'})

# Fix misleading columns

### Procedure type

According to an email from a public employer administrating compranet:

'Si bien en la columna Tipo Procedimiento aparecen Adjudicaciones Directas, la descripción completa es ADJUDICACIÓN DIRECTA POR LICITACIONES PÚBLICAS DESIERTAS, lo que indica que inicialmente el tipo de procedimiento fue una Licitación Pública, que contempla en su cronograma de eventos una fecha de apertura; por lo anterior, en la columna Fecha de apertura se muestra un valor para cada fila aún cuando se trate de un procedimiento de Adjudicación Directa.'

Which means that values under 'Fecha de apertura' / 'Submission deadline date' and procedure_type == direct, are in reality procedures that started as open procedures and since the call was desert, they finished being direct.

I will create another category on procedure type to account for this

In [61]:
contracts_and_procedure.columns

Index(['government_level_c', 'purchasing_unit_name_c', 'file_code_c',
       'file_template_c', 'procedure_number_c', 'country_contract_type_c',
       'contracting_type_c', 'procedure_type_c', 'procedure_venue_c',
       'contract_code_c', 'contract_initial_date_c', 'contract_end_date_c',
       'contract_price_mx_c', 'contract_signature_date_c',
       'supplier_type_alternative_c', 'providers_registry_code_c',
       'supplier_name_c', 'supplier_type_c', 'advertisement_url_c',
       'df_year_c', 'legal_fundament_c', 'RFC_c', 'purchasing_unit_id_c',
       'supply_type_c', 'legal_framework_c', 'supplier_size_c',
       'supplier_name_clean_c', 'legal_fundament_standard_c',
       'legal_fundament_simplified_c', 'contract_year_c', 'file_code_pf',
       'state_pf', 'advertisement_link_pf', 'number_of_tenderers',
       'awards_per_tender', 'advertisement_date', 'submission_deadline_date',
       'award_date'],
      dtype='object')

In [62]:
contracts_and_procedure['procedure_type_c'].value_counts(dropna=False) / len(contracts_and_procedure)

direct            0.748071
open              0.142321
at_least_three    0.099763
other             0.007195
NaN               0.002651
Name: procedure_type_c, dtype: float64

In [63]:
contracts_and_procedure['procedure_type_fixed'] = contracts_and_procedure['procedure_type_c'].copy()
contracts_and_procedure['procedure_type_fixed'] = np.where((contracts_and_procedure['procedure_type_fixed'] == 'direct') & (contracts_and_procedure['submission_deadline_date'].notnull()), 'direct_failed_open', contracts_and_procedure['procedure_type_fixed'])

In [64]:
contracts_and_procedure[contracts_and_procedure['procedure_type_fixed'] == 'direct'][['submission_deadline_date']].dropna()

,submission_deadline_date


In [65]:
contracts_and_procedure['procedure_type_fixed'].value_counts(dropna=False) / len(contracts_and_procedure)

direct_failed_open    0.469168
direct                0.278902
open                  0.142321
at_least_three        0.099763
other                 0.007195
NaN                   0.002651
Name: procedure_type_fixed, dtype: float64

In [66]:
contracts_and_procedure['legal_fundament_standard_simplified_wm'] = contracts_and_procedure['legal_fundament_simplified_c'].fillna('missing')
pd.crosstab(contracts_and_procedure['procedure_type_fixed'], contracts_and_procedure['legal_fundament_standard_simplified_wm'], normalize='index', margins=True)

legal_fundament_standard_simplified_wm,art41,art42,art43,missing
procedure_type_fixed,,,,
at_least_three,0.053793,0.158415,0.083979,0.703813
direct,0.042815,0.005241,0.000129,0.951815
direct_failed_open,0.431142,0.223226,0.010894,0.334739
open,0.000618,0.000316,0.000128,0.998938
other,0.000722,0.000361,0.000000,0.998916
All,0.220263,0.122368,0.013579,0.643790


In [67]:
contracts_and_procedure['legal_fundament_simplified_c'].isnull().sum() / len(contracts_and_procedure)

0.6447342206792128

We can see that the least missing legal justification are in direct_failed_open, which means that 67% (.67/(1-.61) = 1.71 OR) of values in direct failed open are not-missing. The legal justification is higher in procedures that failed to be open.

In [68]:
pd.crosstab(contracts_and_procedure['procedure_type_fixed'], contracts_and_procedure['legal_fundament_standard_simplified_wm'], normalize='columns', margins=True)

legal_fundament_standard_simplified_wm,art41,art42,art43,missing,All
procedure_type_fixed,,,,,
at_least_three,0.024429,0.129495,0.618612,0.109354,0.100028
direct,0.054357,0.011977,0.002654,0.413441,0.279644
direct_failed_open,0.920789,0.858138,0.377390,0.244592,0.470415
open,0.000400,0.000369,0.001343,0.221419,0.142699
other,0.000024,0.000021,0.000000,0.011193,0.007214


Here we can see that direct failed open has the highest concentration of art41 (.78/0.44) and art42 (0.75/0.44), the exceptions for direct procedures. 

Therefore, this reclassification is valid.

# Before exporting

In [69]:
delete_columns = ['file_code_pf', 'advertisement_link_pf']
#drop columns
contracts_and_procedure = contracts_and_procedure.drop(columns=delete_columns)

In [70]:
contracts_and_procedure.columns = [s[:-2] if s.endswith('_c') else s for s in contracts_and_procedure.columns]


In [71]:
(contracts_and_procedure.isnull().sum() / len(contracts_and_procedure)).sort_index(ascending=True)

RFC                                       0.357048
advertisement_date                        0.173678
advertisement_url                         0.000000
award_date                                0.595893
awards_per_tender                         0.664667
contract_code                             0.000000
contract_end_date                         0.000141
contract_initial_date                     0.000000
contract_price_mx                         0.000000
contract_signature_date                   0.591509
contract_year                             0.000000
contracting_type                          0.003249
country_contract_type                     0.029391
df_year                                   0.000000
file_code                                 0.000000
file_template                             0.000340
government_level                          0.000000
legal_framework                           0.029391
legal_fundament                           0.644734
legal_fundament_simplified     

In [72]:
contracts_and_procedure['providers_registry_code'] = contracts_and_procedure['providers_registry_code'].astype(str)  # Ensure the column is of type string
contracts_and_procedure['providers_registry_code'] = contracts_and_procedure['providers_registry_code'].where(~contracts_and_procedure['providers_registry_code'].str.contains('[a-zA-Z]'), np.nan)
#if it finds '.0' replace it with ''
contracts_and_procedure['providers_registry_code'] = contracts_and_procedure['providers_registry_code'].str.replace('.0', '', regex = False)



In [73]:
contracts_and_procedure['providers_registry_code'].isnull().sum() / len(contracts_and_procedure)

0.6378365876856072

In [75]:
contracts_and_procedure['providers_registry_code'] = np.where(contracts_and_procedure['providers_registry_code'].isnull(), -999, contracts_and_procedure['providers_registry_code'])

In [76]:
contracts_and_procedure['providers_registry_code'] = contracts_and_procedure['providers_registry_code'].astype(int)

# Export the dataset

In [77]:
contracts_and_procedure.shape

(2308908, 38)

In [78]:
#export as feather
contracts_and_procedure.to_feather(processed_data / 'mxc11to22_base.feather')
